# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply nearest-centroid classifiers and neighbor graph transformers. 
- Use Neighborhood Components Analysis to learn supervised feature transformations. 
- Evaluate limitations of neighbor methods in high-dimensional and connected workflows. 


## **1. Centroid Methods**

### **1.1. Nearest Centroid Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_01.jpg?v=1784001695" width="250">



>* Summarizes each class with one centroid
>* Classifies by nearest average feature profile

>* Fast, simple, and easy to explain
>* Compares new cases to class averages

>* Scale features and choose distances carefully
>* Works best with compact, average-centered classes



In [ ]:
#@title Python Code - Nearest Centroid Basics

# This example trains a nearest-centroid classifier.
# Centroids summarize each class as one prototype.
# The plot shows predictions and learned centroids.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Create compact two-dimensional classes for easy visualization.
features, labels = make_blobs(
    n_samples=150,
    centers=3,
    cluster_std=1.2,
    random_state=42,
)

# Check that the feature matrix matches the label vector.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Each row needs one class label.")

# Split data so evaluation uses examples unseen during training.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42,
)

# Standardization keeps both feature axes equally important.
model = make_pipeline(StandardScaler(), NearestCentroid())
model.fit(X_train, y_train)

# Predict test labels using the nearest learned class centroid.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Convert centroids back to original feature units for plotting.
scaler = model.named_steps["standardscaler"]
classifier = model.named_steps["nearestcentroid"]
centroids = scaler.inverse_transform(classifier.centroids_)

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))
print("Number of learned centroids:", centroids.shape[0])

# Plot test points and the class prototypes.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_pred,
    cmap="viridis",
    alpha=0.75,
    label="test predictions",
)

# Mark each centroid as a large red X.
ax.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c="red",
    marker="X",
    s=220,
    edgecolor="black",
    label="class centroids",
)

ax.set_title("Nearest Centroid: points classified by closest prototype")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **1.2. Shrunken Centroids**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_02.jpg?v=1784001693" width="250">



>* Shrink centroids to reduce noisy feature effects
>* Emphasize features with meaningful class differences

>* Shrinkage downweights noisy, weak feature differences
>* Regularization improves high-dimensional generalization

>* Highlights key features for clearer classification
>* Tune shrinkage to balance signal and noise



In [ ]:
#@title Python Code - Shrunken Centroids

# This example demonstrates shrunken nearest centroids.
# Shrinkage reduces weak feature differences between classes.
# The output compares accuracy and centroid changes.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import accuracy_score

# Create many features, but only a few are truly informative.
features, labels = make_classification(
    n_samples=240,
    n_features=20,
    n_informative=4,
    n_redundant=0,
    n_repeated=0,
    n_classes=3,
    random_state=42,
)

# Split data so accuracy is measured on unseen examples.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# Validate that the generated data matches the lesson design.
if train_features.shape[1] != 20:
    raise ValueError("Expected exactly 20 features for this lesson.")

# Fit the same centroid classifier with and without shrinkage.
plain_model = NearestCentroid()
plain_model.fit(train_features, train_labels)

shrunk_model = NearestCentroid(shrink_threshold=0.6)
shrunk_model.fit(train_features, train_labels)

# Compare how well each model predicts the held-out test set.
plain_accuracy = accuracy_score(test_labels, plain_model.predict(test_features))
shrunk_accuracy = accuracy_score(test_labels, shrunk_model.predict(test_features))

# Count centroid entries pulled exactly to the overall feature average.
overall_mean = train_features.mean(axis=0)
shrunk_differences = np.abs(shrunk_model.centroids_ - overall_mean)
near_overall_mean = int(np.sum(shrunk_differences < 1e-12))

# Show a compact summary of the shrinkage effect.
print("scikit-learn version:", sklearn.__version__)
print("Plain nearest-centroid accuracy:", round(plain_accuracy, 3))
print("Shrunken-centroid accuracy:", round(shrunk_accuracy, 3))
print("Centroid values shrunk to overall mean:", near_overall_mean)
print("Total centroid values:", shrunk_model.centroids_.size)



### **1.3. Centroid Limitations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_03.jpg?v=1784001697" width="250">



>* One centroid assumes one central class cluster
>* Multimodal classes can be poorly represented

>* Scale features to prevent distance dominance
>* Outliers and mixed data need careful handling

>* Simple centroids miss complex local boundaries
>* Use richer models when structure demands



## **2. Neighbor Graphs**

### **2.1. K Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_01.jpg?v=1784001699" width="250">



>* Connect points to their k nearest neighbors
>* Check whether NCA improves class neighborhoods

>* Graphs reveal meaningful neighbor connections
>* NCA improves class-based local grouping

>* Choose k to balance detail and stability
>* Validate graphs against performance and domain knowledge



In [ ]:
#@title Python Code - K Neighbor Graphs

# Build a small k-neighbor graph.
# Compare raw and NCA-transformed neighborhoods.
# Visualize stronger same-class local connections.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_wine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.neighbors import kneighbors_graph

# Load a small packaged classification dataset.
wine = load_wine()
features = wine.data
target = wine.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split first, so preprocessing learns only from training data.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, stratify=target, random_state=42
)

# Standardize features before distance-based learning.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Learn a supervised two-dimensional feature transformation.
nca = NeighborhoodComponentsAnalysis(n_components=2, random_state=42, max_iter=200)
nca.fit(X_train_scaled, y_train)

# Transform the held-out points into the learned space.
X_test_nca = nca.transform(X_test_scaled)

# Build k-neighbor graphs before and after NCA.
k = 5
raw_graph = kneighbors_graph(X_test_scaled, k, mode="connectivity")
nca_graph = kneighbors_graph(X_test_nca, k, mode="connectivity")

# Measure how often graph edges connect matching labels.
def same_class_edge_rate(graph, labels):
    rows, cols = graph.nonzero()
    matches = labels[rows] == labels[cols]
    return matches.mean()

raw_rate = same_class_edge_rate(raw_graph, y_test)
nca_rate = same_class_edge_rate(nca_graph, y_test)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Raw k-neighbor same-class edge rate: {raw_rate:.2f}")
print(f"NCA k-neighbor same-class edge rate: {nca_rate:.2f}")

# Plot the transformed space and draw selected neighbor edges.
fig, ax = plt.subplots(figsize=(7, 5))
colors = np.array(["tab:blue", "tab:orange", "tab:green"])

rows, cols = nca_graph.nonzero()
for edge_number in range(min(80, len(rows))):
    start = rows[edge_number]
    end = cols[edge_number]
    ax.plot(
        [X_test_nca[start, 0], X_test_nca[end, 0]],
        [X_test_nca[start, 1], X_test_nca[end, 1]],
        color="lightgray",
        linewidth=0.8,
        zorder=1,
    )

for class_id in np.unique(y_test):
    class_mask = y_test == class_id
    ax.scatter(
        X_test_nca[class_mask, 0],
        X_test_nca[class_mask, 1],
        color=colors[class_id],
        label=wine.target_names[class_id],
        s=45,
        zorder=2,
    )

ax.set_title("K-neighbor graph after NCA transformation")
ax.set_xlabel("NCA component 1")
ax.set_ylabel("NCA component 2")
ax.legend(title="Wine class")
plt.show()



### **2.2. Radius Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_02.jpg?v=1784001701" width="250">



>* Connect points within a chosen distance
>* Adapt neighbors to local data density

>* Radius graphs reveal class-focused transformed neighborhoods
>* They diagnose useful learned distance organization

>* Choose radius to match transformed feature scale
>* Validate graphs against noise and uneven density



In [ ]:
#@title Python Code - Radius Neighbor Graphs

# This example builds radius neighbor graphs.
# NCA learns a supervised feature transformation.
# The plot compares neighborhoods before and after.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.neighbors import radius_neighbors_graph
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset with mixed classes.
features, labels = make_classification(
    n_samples=180,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.0,
    random_state=42,
)

# Split first, so scaling and NCA learn only training information.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# Standardize distances before applying neighbor-based methods.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
test_scaled = scaler.transform(test_features)

# Learn a two-dimensional space that keeps same-class neighbors close.
nca = NeighborhoodComponentsAnalysis(
    n_components=2,
    max_iter=200,
    random_state=42,
)
nca.fit(train_scaled, train_labels)

# Transform the held-out data for an honest neighborhood check.
test_transformed = nca.transform(test_scaled)

# Validate that the transformed data has the expected two columns.
if test_transformed.shape[1] != 2:
    raise ValueError("NCA should create exactly two transformed features.")

# Use one radius in each space based on typical local distances.
original_radius = 1.25
transformed_radius = 1.25

# Build radius neighbor graphs for the same test observations.
original_graph = radius_neighbors_graph(
    test_scaled,
    radius=original_radius,
    mode="connectivity",
    include_self=False,
)
transformed_graph = radius_neighbors_graph(
    test_transformed,
    radius=transformed_radius,
    mode="connectivity",
    include_self=False,
)

# Count how many graph edges connect observations from the same class.
def same_class_edge_rate(graph, graph_labels):
    row_indices, column_indices = graph.nonzero()
    edge_count = len(row_indices)
    if edge_count == 0:
        return 0.0, 0
    same_class = graph_labels[row_indices] == graph_labels[column_indices]
    return float(np.mean(same_class)), edge_count

original_rate, original_edges = same_class_edge_rate(original_graph, test_labels)
transformed_rate, transformed_edges = same_class_edge_rate(
    transformed_graph,
    test_labels,
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original radius graph: {original_edges} edges, {original_rate:.2f} same-class")
print(f"NCA radius graph: {transformed_edges} edges, {transformed_rate:.2f} same-class")

# Plot the transformed space and draw a few radius graph connections.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    test_transformed[:, 0],
    test_transformed[:, 1],
    c=test_labels,
    cmap="viridis",
    s=45,
    edgecolor="black",
)

# Draw only a small deterministic sample of edges for readability.
rows, cols = transformed_graph.nonzero()
edge_limit = min(35, len(rows))
for edge_number in range(edge_limit):
    start = rows[edge_number]
    end = cols[edge_number]
    ax.plot(
        [test_transformed[start, 0], test_transformed[end, 0]],
        [test_transformed[start, 1], test_transformed[end, 1]],
        color="gray",
        alpha=0.35,
        linewidth=1,
    )

ax.set_title("Radius Neighbor Graph After NCA")
ax.set_xlabel("NCA feature 1")
ax.set_ylabel("NCA feature 2")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()



### **2.3. KNeighborsTransformer**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_03.jpg?v=1784001703" width="250">



>* Turns nearest neighbors into sparse graphs
>* Shows NCA-learned class similarities clearly

>* Build reusable neighbor graphs for later models
>* Store connections or distances between similar samples

>* Feature quality shapes meaningful neighbor graphs
>* Tune neighbor count to balance structure



In [ ]:
#@title Python Code - KNeighborsTransformer

# Build a neighbor graph after supervised feature learning.
# Compare original and NCA transformed local neighborhoods.
# Expect stronger same-class connections after transformation.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_wine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsTransformer
from sklearn.neighbors import NeighborhoodComponentsAnalysis

# Load a small packaged classification dataset.
wine = load_wine()
features = wine.data
target = wine.target

# Validate the basic dataset shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split first, so scaling and NCA learn only from training data.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, stratify=target, random_state=42
)

# Standardize features before distance-based neighbor methods.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Learn a supervised transformation that favors same-class neighbors.
nca = NeighborhoodComponentsAnalysis(n_components=2, random_state=42, max_iter=200)
nca.fit(X_train_scaled, y_train)

# Transform the test set into the learned two-dimensional space.
X_test_nca = nca.transform(X_test_scaled)

# Build neighbor graphs before and after the supervised transformation.
graph_original = KNeighborsTransformer(n_neighbors=5, mode="connectivity")
graph_nca = KNeighborsTransformer(n_neighbors=5, mode="connectivity")

original_neighbors = graph_original.fit_transform(X_test_scaled)
nca_neighbors = graph_nca.fit_transform(X_test_nca)

# Measure how often graph edges connect samples of the same class.
def same_class_edge_rate(graph, labels):
    rows, cols = graph.nonzero()
    keep = rows != cols
    rows = rows[keep]
    cols = cols[keep]
    return np.mean(labels[rows] == labels[cols])

original_rate = same_class_edge_rate(original_neighbors, y_test)
nca_rate = same_class_edge_rate(nca_neighbors, y_test)

print("scikit-learn version:", sklearn.__version__)
print("Same-class edge rate before NCA:", round(original_rate, 3))
print("Same-class edge rate after NCA:", round(nca_rate, 3))

# Plot the learned space where the neighbor graph was built.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(X_test_nca[:, 0], X_test_nca[:, 1], c=y_test, cmap="viridis")

ax.set_title("Wine test samples after NCA")
ax.set_xlabel("NCA component 1")
ax.set_ylabel("NCA component 2")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()



## **3. Metric Learning Limits**

### **3.1. NCA Feature Transformation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_01.jpg?v=1784001705" width="250">



>* NCA reshapes distances to improve class neighborhoods
>* Poor labels can distort learned transformations

>* NCA can overfit labeled training data
>* Validate high-dimensional transformations carefully

>* NCA changes downstream neighbor relationships
>* Check learned spaces for fairness and stability



In [ ]:
#@title Python Code - NCA Feature Transformation

# This example shows NCA feature transformation.
# It compares neighbor accuracy before and after NCA.
# It highlights overfitting risk in high dimensions.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create many distracting features around a small signal.
features, labels = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=3,
    n_redundant=2,
    n_classes=3,
    random_state=42,
)

# Check that the generated data has the expected shape.
if features.shape != (500, 20):
    raise ValueError("Expected 500 rows and 20 features.")

# Split once so test accuracy measures generalization.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# A plain neighbor model uses all original distances equally.
plain_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)

# NCA learns a supervised transformation before neighbor voting.
nca_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("nca", NeighborhoodComponentsAnalysis(random_state=42, max_iter=80)),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)

# Fit both workflows only on the training data.
plain_knn.fit(train_features, train_labels)
nca_knn.fit(train_features, train_labels)

# Compare training and test scores to reveal possible overfitting.
plain_train = plain_knn.score(train_features, train_labels)
plain_test = plain_knn.score(test_features, test_labels)
nca_train = nca_knn.score(train_features, train_labels)
nca_test = nca_knn.score(test_features, test_labels)

print("scikit-learn version:", sklearn.__version__)
print("Plain KNN train/test accuracy:", round(plain_train, 3), round(plain_test, 3))
print("NCA + KNN train/test accuracy:", round(nca_train, 3), round(nca_test, 3))
print("Lesson: NCA can help distances, but test accuracy checks limits.")



### **3.2. NCA KNN Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_02.jpg?v=1784001707" width="250">



>* NCA reshapes features for better neighbor grouping
>* KNN classifies using transformed similar cases

>* NCA can overfit noisy high-dimensional data
>* Validate within splits to prevent leakage

>* Learned distances can be hard to explain
>* Retraining, indexing, and drift add complexity



In [ ]:
#@title Python Code - NCA KNN Pipeline

# This example compares KNN with an NCA pipeline.
# It shows validation without leaking label information.
# Results highlight overfitting risk in noisy dimensions.

import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a small high-dimensional classification dataset.
features, labels = make_classification(
    n_samples=500,
    n_features=25,
    n_informative=5,
    n_redundant=0,
    n_classes=3,
    random_state=42,
)

# Check that the generated data match the intended lesson.
if features.shape != (500, 25):
    raise ValueError("Expected 500 rows and 25 features.")

# Split before fitting any supervised transformation.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# Plain KNN uses distances in the original scaled space.
plain_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)

# NCA learns a supervised space only from training data.
nca_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("nca", NeighborhoodComponentsAnalysis(random_state=42, max_iter=80)),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)

# Fit both workflows on the same training split.
plain_knn.fit(train_features, train_labels)
nca_knn.fit(train_features, train_labels)

# Compare training and test accuracy to reveal generalization limits.
plain_train = plain_knn.score(train_features, train_labels)
plain_test = plain_knn.score(test_features, test_labels)
nca_train = nca_knn.score(train_features, train_labels)
nca_test = nca_knn.score(test_features, test_labels)

print("scikit-learn version:", sklearn.__version__)
print("Plain KNN train/test:", round(plain_train, 3), round(plain_test, 3))
print("NCA + KNN train/test:", round(nca_train, 3), round(nca_test, 3))
print("NCA can improve training fit, but test accuracy is the key check.")



### **3.3. High Dimensional Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_03.jpg?v=1784001709" width="250">



>* High dimensions make nearness less meaningful
>* Noisy features can hide true similarity

>* NCA highlights useful features for better neighbors
>* Limited data can cause overfitting

>* Unreliable distances can mislead entire workflows
>* Validate neighborhood stability beyond accuracy



In [ ]:
#@title Python Code - High Dimensional Limits

# This example shows high-dimensional neighbor limits.
# Extra noisy features can weaken distance meaning.
# The plot compares accuracy as dimensions grow.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# We keep only two informative features throughout.
feature_counts = [2, 5, 10, 20, 40]
accuracies = []

# Each dataset adds more noisy features around the same signal.
for feature_count in feature_counts:
    X, y = make_classification(
        n_samples=800,
        n_features=feature_count,
        n_informative=2,
        n_redundant=0,
        n_repeated=0,
        n_classes=2,
        class_sep=1.2,
        random_state=42,
    )

    if X.shape[1] != feature_count:
        raise ValueError("Generated feature count did not match the lesson.")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=42,
    )

    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
    model.fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

# The printed lines summarize the key lesson.
print("scikit-learn version:", sklearn.__version__)
print("Only 2 features are informative; the rest are noise.")
print("Accuracy with 2 features:", round(accuracies[0], 3))
print("Accuracy with 40 features:", round(accuracies[-1], 3))

# The plot shows how noisy dimensions can hurt neighbors.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(feature_counts, accuracies, marker="o")
ax.set_title("KNN accuracy as noisy dimensions increase")
ax.set_xlabel("Total number of features")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>


In this lecture, you learned to:
- Apply nearest-centroid classifiers and neighbor graph transformers. 
- Use Neighborhood Components Analysis to learn supervised feature transformations. 
- Evaluate limitations of neighbor methods in high-dimensional and connected workflows. 

In the next Module (Module 14), we will go over 'Gaussian Cross'